# Notebook 02 — Modelling Pipeline

**Project**: Credit Card Fraud Detection  
**Goal**: Train, compare, tune and evaluate models. Select threshold via cost model. Explain predictions with SHAP.

This notebook orchestrates the scripts in `src/`. Run cells in order.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')

ROOT = pathlib.Path().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

STYLE = {
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#c9d1d9', 'grid.color': '#21262d',
    'grid.linewidth': 0.6, 'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d', 'font.family': 'DejaVu Sans', 'font.size': 11,
}
plt.rcParams.update(STYLE)

def save(fig, name, dpi=150):
    path = FIGURES_DIR / name
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f'Saved: {path}')

print('Imports OK')

## 1. Load Data and Split

In [ ]:
from data import load_raw, audit, print_audit, drop_duplicates, stratified_split, time_ordered_split
from features import get_feature_matrix, build_preprocessor, build_pipeline, FEATURE_COLS

df = load_raw()
info = audit(df)
print_audit(info)

df = drop_duplicates(df)
train_df, val_df, test_df = stratified_split(df, val_size=0.15, test_size=0.15, random_state=42)

X_train, y_train = get_feature_matrix(train_df)
X_val,   y_val   = get_feature_matrix(val_df)
X_test,  y_test  = get_feature_matrix(test_df)

print(f'\nTrain: {len(X_train):,}   Val: {len(X_val):,}   Test: {len(X_test):,}')

## 2. Why Accuracy is Not Enough

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, average_precision_score

dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_dummy_pred = dummy.predict(X_train)
y_dummy_prob = dummy.predict_proba(X_train)[:, 1]

dummy_acc = accuracy_score(y_train, y_dummy_pred)
dummy_ap  = average_precision_score(y_train, y_dummy_prob)

print(f'Dummy (always Genuine) accuracy : {dummy_acc:.4f}  ({dummy_acc*100:.2f}%)')
print(f'Dummy average precision (PR-AUC): {dummy_ap:.4f}')
print()
print('→ 99.83% accuracy by predicting everything as genuine.')
print('→ PR-AUC of ~0.0017 (fraud rate) — tells the real story.')
print()
print('This is why we use PR-AUC as our primary metric.')

## 3. Cross-Validation: All Model Candidates

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, make_scorer, fbeta_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb

n0 = (y_train == 0).sum()
n1 = (y_train == 1).sum()
class_ratio = n0 / n1
print(f'Class ratio (genuine/fraud): {class_ratio:.1f}')

ap_scorer  = make_scorer(average_precision_score, response_method='predict_proba', pos_label=1)
roc_scorer = make_scorer(roc_auc_score, response_method='predict_proba')
f2_scorer  = make_scorer(fbeta_score, beta=2, pos_label=1, zero_division=0)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = {'avg_precision': ap_scorer, 'roc_auc': roc_scorer, 'f2': f2_scorer}

def evaluate(label, pipeline):
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    return {
        'Model': label,
        'PR_AUC': f"{np.mean(scores['test_avg_precision']):.4f} ± {np.std(scores['test_avg_precision']):.4f}",
        'ROC_AUC': f"{np.mean(scores['test_roc_auc']):.4f} ± {np.std(scores['test_roc_auc']):.4f}",
        'F2': f"{np.mean(scores['test_f2']):.4f} ± {np.std(scores['test_f2']):.4f}",
        '_pr_mean': np.mean(scores['test_avg_precision']),
    }

print('Running cross-validation (this takes a few minutes) ...')

In [ ]:
results = []

# Baselines
results.append(evaluate('LogReg (balanced)', build_pipeline(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))))

# Random Forest variants
results.append(evaluate('RF (no resamp.)', build_pipeline(
    RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42))))
results.append(evaluate('RF (balanced)', build_pipeline(
    RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=42))))
results.append(evaluate('RF + SMOTE', ImbPipeline([
    ('pre', build_preprocessor()), ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42))])))  
results.append(evaluate('RF + RUS', ImbPipeline([
    ('pre', build_preprocessor()), ('rus', RandomUnderSampler(random_state=42)),
    ('clf', RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42))])))

# XGBoost variants
results.append(evaluate('XGB (no resamp.)', build_pipeline(
    xgb.XGBClassifier(n_estimators=200, eval_metric='aucpr', n_jobs=-1, verbosity=0, random_state=42))))
results.append(evaluate('XGB (scale_pos_weight)', build_pipeline(
    xgb.XGBClassifier(n_estimators=200, scale_pos_weight=class_ratio,
                       eval_metric='aucpr', n_jobs=-1, verbosity=0, random_state=42))))
results.append(evaluate('XGB + SMOTE', ImbPipeline([
    ('pre', build_preprocessor()), ('smote', SMOTE(random_state=42)),
    ('clf', xgb.XGBClassifier(n_estimators=200, eval_metric='aucpr', n_jobs=-1, verbosity=0, random_state=42))])))

results_df = pd.DataFrame(results)
display_df = results_df.drop(columns=['_pr_mean'])
display_df

## 4. Hyperparameter Tuning (XGBoost)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

print('Tuning XGBoost with RandomizedSearchCV (30 iterations, 5-fold CV) ...')
print('This may take 10-20 minutes on CPU.')

xgb_pipe = build_pipeline(
    xgb.XGBClassifier(eval_metric='aucpr', n_jobs=-1, verbosity=0, random_state=42)
)

param_dist = {
    'classifier__n_estimators':     [200, 300, 500],
    'classifier__max_depth':        [3, 4, 5, 6],
    'classifier__learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'classifier__subsample':        [0.6, 0.7, 0.8, 0.9],
    'classifier__colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'classifier__min_child_weight': [1, 3, 5],
    'classifier__gamma':            [0, 0.1, 0.3],
    'classifier__scale_pos_weight': [1, class_ratio * 0.5, class_ratio],
}

search = RandomizedSearchCV(
    xgb_pipe, param_distributions=param_dist,
    n_iter=30, scoring='average_precision',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1, random_state=42, refit=True, verbose=1,
)
search.fit(X_train, y_train)

print(f'Best CV PR-AUC: {search.best_score_:.4f}')
print(f'Best params: {search.best_params_}')
best_model = search.best_estimator_

In [ ]:
# Evaluate tuned model
tuned_res = evaluate('XGB (tuned)', best_model)
results_df = pd.concat([results_df, pd.DataFrame([tuned_res])], ignore_index=True)

# Save results CSV
import pathlib
results_df.drop(columns=['_pr_mean']).to_csv(ROOT / 'reports' / 'results.csv', index=False)
print('Results saved to reports/results.csv')
results_df.drop(columns=['_pr_mean'])

## 5. Precision-Recall Curves on Validation Set

In [ ]:
from sklearn.metrics import precision_recall_curve

# Compare key models on validation set
val_models = [
    ('LogReg (balanced)', build_pipeline(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))),
    ('RF (balanced)', build_pipeline(RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=42))),
    ('XGB (tuned)', best_model),
]

COLORS = ['#58a6ff', '#3fb950', '#f0883e', '#d2a8ff', '#f85149']

fig, ax = plt.subplots(figsize=(9, 6))
baseline = y_val.mean()
ax.axhline(baseline, ls='--', color='#8b949e', lw=1.5, label=f'No-skill ({baseline:.4f})')

for i, (label, pipe) in enumerate(val_models):
    pipe.fit(X_train, y_train)  # fit on train, eval on val
    y_prob = pipe.predict_proba(X_val)[:, 1]
    prec, rec, _ = precision_recall_curve(y_val, y_prob)
    ap = average_precision_score(y_val, y_prob)
    ax.plot(rec, prec, lw=2, color=COLORS[i], label=f'{label} (AP={ap:.3f})')

ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — Validation Set')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.4)
save(fig, 'pr_curves_validation.png')
plt.show()

## 6. Threshold Selection — Cost Model

**Assumptions (ILLUSTRATIVE — not real bank figures):**
- Cost of missed fraud = 1.0 × transaction Amount (bank absorbs the loss)
- Cost of false alarm = 5.0 currency units (analyst review time)

We sweep thresholds on the **validation set** and pick the one minimising total cost.

In [ ]:
FP_COST = 5.0      # cost per false alarm
FN_MULT = 1.0      # multiplier on Amount for missed fraud

from sklearn.metrics import confusion_matrix

y_val_prob = best_model.predict_proba(X_val)[:, 1]
amounts_val = val_df['Amount'].values

thresholds = np.linspace(0.01, 0.99, 500)
costs = []
for t in thresholds:
    yp = (y_val_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, yp, labels=[0,1]).ravel()
    fn_mask = (y_val.values == 1) & (yp == 0)
    cost = FN_MULT * amounts_val[fn_mask].sum() + FP_COST * fp
    costs.append(cost)

costs = np.array(costs)
opt_idx = np.argmin(costs)
opt_t   = float(thresholds[opt_idx])
print(f'Optimal threshold: {opt_t:.4f}')
print(f'Min cost (val):    {costs[opt_idx]:,.2f}')

# Save threshold
with open(ROOT / 'reports' / 'optimal_threshold.txt', 'w') as f:
    f.write(str(opt_t))
print('Threshold saved.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
cost_05 = costs[np.argmin(np.abs(thresholds - 0.5))]
ax.plot(thresholds, costs, lw=2, color='#58a6ff', label='Total cost (val)')
ax.axvline(opt_t, ls='--', color='#3fb950', label=f'Optimal t={opt_t:.3f}  cost={costs[opt_idx]:,.0f}')
ax.axvline(0.5,   ls='--', color='#f85149', label=f't=0.50  cost={cost_05:,.0f}')
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Total Expected Cost')
ax.set_title('Cost vs Threshold (Validation Set)\n[ILLUSTRATIVE cost assumptions]')
ax.legend(); ax.grid(True, alpha=0.4)
save(fig, 'cost_curve.png')
plt.show()

## 7. Final Evaluation on Test Set

The test set is used exactly once here.

In [ ]:
import joblib
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    classification_report, f1_score, fbeta_score,
    precision_score, recall_score,
)

# Refit on train + val
X_trainval = pd.concat([X_train, X_val], ignore_index=True)
y_trainval = pd.concat([y_train, y_val], ignore_index=True)
best_model.fit(X_trainval, y_trainval)

# Save model
import pathlib
(ROOT / 'models').mkdir(exist_ok=True)
joblib.dump(best_model, ROOT / 'models' / 'final_model.joblib')
print('Model saved.')

# Evaluate on test
y_test_prob = best_model.predict_proba(X_test)[:, 1]
y_test_opt  = (y_test_prob >= opt_t).astype(int)
y_test_05   = (y_test_prob >= 0.5).astype(int)

pr_auc  = average_precision_score(y_test, y_test_prob)
roc_auc = roc_auc_score(y_test, y_test_prob)

print(f'\n=== FINAL TEST SET RESULTS ===')
print(f'PR-AUC  (Average Precision): {pr_auc:.4f}')
print(f'ROC-AUC                    : {roc_auc:.4f}')

for name, yp in [('t=0.50', y_test_05), (f't={opt_t:.3f} (optimal)', y_test_opt)]:
    print(f'\n  Threshold {name}')
    print(f'  Precision: {precision_score(y_test, yp, zero_division=0):.4f}')
    print(f'  Recall   : {recall_score(y_test, yp, zero_division=0):.4f}')
    print(f'  F1       : {f1_score(y_test, yp, zero_division=0):.4f}')
    print(f'  F2       : {fbeta_score(y_test, yp, beta=2, zero_division=0):.4f}')

print('\nDetailed report (optimal threshold):')
print(classification_report(y_test, y_test_opt,
      target_names=['Genuine', 'Fraud'], zero_division=0))

In [ ]:
# Cost comparison on test set
amounts_test = test_df['Amount'].values

def total_cost(y_true, y_pred, amounts):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    fn_mask = (np.array(y_true) == 1) & (np.array(y_pred) == 0)
    return FN_MULT * amounts[fn_mask].sum() + FP_COST * fp

cost_opt  = total_cost(y_test, y_test_opt,  amounts_test)
cost_base = total_cost(y_test, y_test_05,   amounts_test)

print(f'Test cost @ optimal threshold : {cost_opt:,.2f}')
print(f'Test cost @ t=0.50            : {cost_base:,.2f}')
print(f'Cost reduction                : {(cost_base - cost_opt) / cost_base * 100:.1f}%')

## 8. SHAP Explanations

In [ ]:
import shap
from features import SCALE_COLS, V_COLS

preprocessor = best_model.named_steps['preprocessor']
classifier   = best_model.named_steps['classifier']

X_test_prep = preprocessor.transform(X_test)
col_names   = SCALE_COLS + V_COLS
X_test_prep_df = pd.DataFrame(X_test_prep, columns=col_names)

print('Computing SHAP values ...')
explainer   = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_test_prep_df)

sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# Summary plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(sv, X_test_prep_df, max_display=20, show=False)
plt.title('SHAP Summary Plot — Feature Importance', color='#c9d1d9', pad=12)
plt.tight_layout()
save(plt.gcf(), 'shap_summary.png')
plt.show()

print('\nHonesty note: V1-V28 are PCA-anonymized. SHAP values show')
print('statistical importance only — no business-level interpretation is possible.')

## 9. Summary

| Metric | Value |
|--------|-------|
| Primary metric | PR-AUC (Average Precision) |
| Best model | XGBoost (tuned) |
| PR-AUC (CV mean ± std) | *see results.csv* |
| PR-AUC (test) | *see cell above* |
| Optimal threshold | *see cost curve* |
| Cost reduction vs t=0.5 | *see cell above* |

### Limitations
- V1-V28 are anonymized PCA features — business interpretation is impossible.
- Only two days of data from 2013 European cardholders — concept drift likely over time.
- Only 492 positives — metric variance is high (wide confidence intervals).
- Cost model uses illustrative assumptions, not real bank figures.
- No geographic, merchant-category or cardholder history features.